# PickLoop 미등록 상품 사진 재시험
Colab 또는 Kaggle에서 위에서부터 실행하세요. CPU와 인터넷 연결을 사용합니다. 기본값은 블러드오렌지와 오렌지·망고 두 장입니다. 16장을 모두 읽으려면 두 번째 코드 셀에서 `FULL_SET = True`로 바꾸세요.

원본의 SHA-256을 확인한 후 공개 데모의 파일 선택창에 사진을 입력합니다. 결과에는 실제로 표시된 상품명·미등록 근거·주문 판정과 브라우저 버전을 남깁니다. 두 사진은 개선에 사용한 원본입니다. GPU 환경에서 Isaac Sim을 실행하는 노트북은 [별도로 열 수 있습니다](https://colab.research.google.com/github/sokldjs554/pickloop/blob/main/docs/experiments/isaac-sim/notebook.ipynb).

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "playwright>=1.51,<2"], check=True)
subprocess.run([sys.executable, "-m", "playwright", "install", "--with-deps", "chromium"], check=True)

In [ ]:
import csv
import hashlib
import json
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path
from urllib.request import Request, urlopen
from playwright.async_api import async_playwright

FULL_SET = False
site = "https://pickloop-robot-ops.ykh13571.chatgpt.site"
source = "https://raw.githubusercontent.com/sokldjs554/pickloop/346772d4a6f7244bad27726f5aeab6dd0af142be/docs/experiments/side-unknown-challenge"
focus = {"U29-66ab1ecf5211d2c0", "U29-82be32458fb0d515"}

def fetch(url):
    with urlopen(Request(url, headers={"User-Agent": "PickLoop-study/1.0"}), timeout=60) as response:
        return response.read()

labels = list(csv.DictReader(StringIO(fetch(source + "/labels.csv").decode())))
inputs = [row for row in labels if FULL_SET or row["id"] in focus]
expected = {"package-reader.js": "b75eea091e445607694f37828546be2c9353f9978858b864a6b0071999fd53e7", "outside-title.js": "b5ce2675b1a925c2f92b514c92079f3ec564fd0ce1173f44a70a63e5c8ac00cc"}
out = Path("pickloop-photo-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f") + ".jsonl")
print(len(inputs), "장", out)

In [ ]:
observations = []
async with async_playwright() as playwright:
    async with await playwright.chromium.launch() as browser:
        page = await browser.new_page()
        await page.goto(site + "/recognition", wait_until="domcontentloaded")
        for name, digest in expected.items():
            actual = await page.evaluate("""async path => {
                const response = await fetch(path, {cache: 'no-store'});
                if (!response.ok) throw new Error('HTTP ' + response.status);
                const hash = await crypto.subtle.digest('SHA-256', await response.arrayBuffer());
                return Array.from(new Uint8Array(hash), b => b.toString(16).padStart(2, '0')).join('');
            }""", "/" + name)
            if actual != digest:
                raise RuntimeError("공개 판독기 버전이 변경되었습니다. " + name)
        for row in inputs:
            raw = fetch(row["image_url"])
            if hashlib.sha256(raw).hexdigest() != row["sha256"]:
                raise ValueError("원본 사진이 변경되었습니다. " + row["id"])
            png = raw.startswith(bytes.fromhex("89504e47"))
            await page.locator("#package-photo").set_input_files({
                "name": "photo.png" if png else "photo.jpg",
                "mimeType": "image/png" if png else "image/jpeg", "buffer": raw,
            })
            await page.locator('#package-reader [type="submit"]').click()
            await page.wait_for_function(
                "() => !document.querySelector('#package-reader').hasAttribute('aria-busy') && !document.querySelector('#package-result').hidden",
                timeout=135000,
            )
            if "마쳤습니다" not in await page.locator("#package-status").inner_text():
                raise RuntimeError("판독이 끝나지 않았습니다. " + row["id"])
            identity = page.locator("#package-identity")
            result = {
                "id": row["id"], "sha256": row["sha256"], "chromium": browser.version,
                "product": await identity.get_attribute("data-product") or None,
                "display": await identity.inner_text(),
                "optical": await page.locator("#package-barcode").inner_text(),
            }
            with out.open("a", encoding="utf-8") as f:
                print(json.dumps(result, ensure_ascii=False), file=f)
            observations.append(result)
            print(row["product"], "→", result["display"])

In [ ]:
outside = sum(any(s in row["display"] for s in ("미등록", "목록 밖")) for row in observations)
named = sum(bool(row["product"]) for row in observations)
ordered = sum(row["optical"].startswith("상품 확정 ·") for row in observations)
assert len(observations) == len(inputs)
print(f"{len(inputs)}장 중 미등록 근거 {outside}장 · 등록 이름 오인 {named}장 · 잘못된 주문 {ordered}장")
print(out.resolve())